# VQE Mejorado: Reproducibilidad y Modelo Continuo (F98Y)

Este notebook complementa `S-aureus-F98Y.ipynb`.
- Añade ejecución reproducible (semilla fija) para VQE.
- Introduce un modelo continuo MJ-Ising para suavizar la optimización.
- Genera gráficos mejorados: superposición de convergencias y comparaciones avanzadas.

Recomendación: ejecuta primero `S-aureus-F98Y.ipynb` hasta la celda de VQE para disponer de variables como `hamiltonian_wt`, `hamiltonian_mut`, `fragment_wt`, `fragment_mut`, `results_wt`, `results_mut`, `energy_wt`, `energy_mut`, y `simulator`/`backend_real`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from qiskit_algorithms.optimizers import COBYLA
except Exception:
    from qiskit.algorithms.optimizers import COBYLA

# Subclase que añade semilla (reproducible) y modelo continuo
try:
    base_cls = ImprovedVQE
except NameError:
    # Fallback mínimo por si no existe ImprovedVQE aún
    class _Base:
        def __init__(self, backend=None):
            self.backend = backend
            self.history = []
            self.iteration = 0
        def evaluate_energy_with_hamiltonian(self, params, hamiltonian_terms, mj_potential):
            n_residues = len(set([t['residues'][0] for t in hamiltonian_terms] + [t['residues'][1] for t in hamiltonian_terms]))
            spins = np.sign(np.tanh(params[:n_residues]))
            spins[spins == 0] = 1
            energy = 0.0
            for term in hamiltonian_terms:
                i, j = term['residues']
                J_ij = term['coefficient']
                if i < len(spins) and j < len(spins):
                    energy += J_ij * spins[i] * spins[j]
            self.history.append(energy)
            return energy
    base_cls = _Base

class ImprovedVQERepro(base_cls):
    def run(self, hamiltonian_terms, sequence, mj_potential=None, max_iter=50, seed=42):
        n_residues = len(sequence)
        rng = np.random.default_rng(seed)
        x0 = rng.normal(0, 0.5, size=n_residues)
        optimizer = COBYLA(maxiter=max_iter)
        result = optimizer.minimize(
            fun=lambda p: self.evaluate_energy_with_hamiltonian(p, hamiltonian_terms, mj_potential),
            x0=x0
        )
        return {
            'energy': result.fun,
            'history': self.history,
            'iterations': getattr(self, 'iteration', None),
            'params': result.x,
            'success': True
        }

    def evaluate_energy_continuous(self, params, hamiltonian_terms):
        n_residues = len(set([t['residues'][0] for t in hamiltonian_terms] + [t['residues'][1] for t in hamiltonian_terms]))
        m = np.tanh(params[:n_residues])
        energy = 0.0
        for term in hamiltonian_terms:
            i, j = term['residues']
            J_ij = term['coefficient']
            if i < len(m) and j < len(m):
                energy += J_ij * m[i] * m[j]
        self.history.append(energy)
        return energy

    def run_continuous(self, hamiltonian_terms, sequence, max_iter=50, seed=123):
        n_residues = len(sequence)
        rng = np.random.default_rng(seed)
        x0 = rng.normal(0, 0.5, size=n_residues)
        optimizer = COBYLA(maxiter=max_iter)
        result = optimizer.minimize(
            fun=lambda p: self.evaluate_energy_continuous(p, hamiltonian_terms),
            x0=x0
        )
        return {
            'energy': result.fun,
            'history': self.history,
            'iterations': getattr(self, 'iteration', None),
            'params': result.x,
            'success': True
        }

print("✅ Subclase ImprovedVQERepro lista (semilla y modelo continuo).")


In [ ]:
# Gráfico: superposición de convergencias (WT vs Mutante)
def smooth(y, w=5):
    if len(y) < w:
        return y
    return np.convolve(y, np.ones(w)/w, mode='valid')

# Obtener resultados: usar existentes o calcular reproducibles
wt_ok = 'results_wt' in globals() and isinstance(results_wt, dict)
mut_ok = 'results_mut' in globals() and isinstance(results_mut, dict)

if not (wt_ok and mut_ok):
    missing = []
    for name in ['hamiltonian_wt','hamiltonian_mut','fragment_wt','fragment_mut']:
        if name not in globals():
            missing.append(name)
    if missing:
        print("⚠️ Faltan variables base: ", missing)
        print("Ejecuta primero S-aureus-F98Y.ipynb (celdas de VQE).")
    else:
        backend = None
        if 'backend_real' in globals():
            backend = backend_real
        elif 'simulator' in globals():
            backend = simulator
        vqe_wt = ImprovedVQERepro(backend)
        vqe_mut = ImprovedVQERepro(backend)
        results_wt = vqe_wt.run(hamiltonian_terms=hamiltonian_wt['terms'], sequence=fragment_wt['sequence'], mj_potential=None, max_iter=50, seed=42)
        results_mut = vqe_mut.run(hamiltonian_terms=hamiltonian_mut['terms'], sequence=fragment_mut['sequence'], mj_potential=None, max_iter=50, seed=42)
        print("✅ Resultados reproducibles calculados (seed=42).")

if wt_ok or ('results_wt' in globals()):
    fig, ax = plt.subplots(figsize=(10,6))
    ax.plot(results_wt['history'], color='#2980b9', alpha=0.5, label='WT bruto')
    ax.plot(results_mut['history'], color='#c0392b', alpha=0.5, label='Mutante bruto')
    ax.plot(smooth(results_wt['history']), color='#1f618d', linewidth=2, label='WT suavizado')
    ax.plot(smooth(results_mut['history']), color='#922b21', linewidth=2, label='Mutante suavizado')
    ax.axhline(results_wt['energy'], color='#2980b9', linestyle='--', label=f"WT óptimo {results_wt['energy']:.2f} kT")
    ax.axhline(results_mut['energy'], color='#c0392b', linestyle='--', label=f"Mut óptimo {results_mut['energy']:.2f} kT")
    ax.set_xlabel('Iteración')
    ax.set_ylabel('Energía (kT)')
    ax.set_title('Convergencia VQE (WT vs Mutante)')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.savefig('vqe_convergence_overlay_f98y.png', dpi=150)
    plt.show()
    print('OK - vqe_convergence_overlay_f98y.png')


In [ ]:
# Ejecutar modelo continuo y comparación
can_cont = all(name in globals() for name in ['hamiltonian_wt','hamiltonian_mut','fragment_wt','fragment_mut'])
if can_cont:
    backend = None
    if 'backend_real' in globals():
        backend = backend_real
    elif 'simulator' in globals():
        backend = simulator
    vqe_wt_cont = ImprovedVQERepro(backend)
    vqe_mut_cont = ImprovedVQERepro(backend)
    res_wt_cont = vqe_wt_cont.run_continuous(hamiltonian_wt['terms'], fragment_wt['sequence'], max_iter=50, seed=7)
    res_mut_cont = vqe_mut_cont.run_continuous(hamiltonian_mut['terms'], fragment_mut['sequence'], max_iter=50, seed=7)
    delta_cont = res_mut_cont['energy'] - res_wt_cont['energy']
    # Energía clásica (si existe)
    e_wt = energy_wt if 'energy_wt' in globals() else None
    e_mut = energy_mut if 'energy_mut' in globals() else None
    labels = ['WT (F98)', 'Mutante (Y98)']
    x = np.arange(2)
    width = 0.35
    plt.figure(figsize=(8,6))
    plt.bar(x - width/2, [res_wt_cont['energy'], res_mut_cont['energy']], width, color='#2ecc71', edgecolor='black', label='VQE continuo')
    if e_wt is not None and e_mut is not None:
        plt.bar(x + width/2, [e_wt, e_mut], width, color='#95a5a6', edgecolor='black', label='Clásico MJ')
    plt.xticks(x, labels)
    plt.ylabel('Energía (kT)')
    plt.title('Comparación MJ clásico vs MJ-Ising continuo (F98Y)')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.text(x[1], (res_mut_cont['energy']+res_wt_cont['energy'])/2, f'Δ_cont={delta_cont:+.2f}', ha='center', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.6))
    plt.tight_layout()
    plt.savefig('vqe_continuous_comparison_f98y.png', dpi=150)
    plt.show()
    print('OK - vqe_continuous_comparison_f98y.png')
else:
    print('⚠️ No se pudo ejecutar el modelo continuo: faltan variables base. Ejecuta el notebook principal.')


In [ ]:
# Gráfico comparativo avanzado (Clásico vs VQE discreto vs VQE continuo)
ready = all(name in globals() for name in ['results_wt','results_mut']) and can_cont
if ready:
    labels = ['WT (F98)', 'Mutante (Y98)']
    x = np.arange(2)
    width = 0.3
    fig, ax = plt.subplots(figsize=(12,7))
    e_wt = energy_wt if 'energy_wt' in globals() else 0
    e_mut = energy_mut if 'energy_mut' in globals() else 0
    b1 = ax.bar(x - width, [e_wt, e_mut], width, label='Clásico MJ', color='#7f8c8d', edgecolor='black')
    b2 = ax.bar(x, [results_wt['energy'], results_mut['energy']], width, label='VQE (discreto)', color='#3498db', edgecolor='black')
    b3 = ax.bar(x + width, [res_wt_cont['energy'], res_mut_cont['energy']], width, label='VQE (continuo)', color='#2ecc71', edgecolor='black')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Energía (kT)')
    ax.set_title('Comparación de métodos: MJ clásico vs VQE (discreto/continuo)')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='best')
    # anotaciones
    for bars in (b1, b2, b3):
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2., h, f'{h:.2f}', ha='center', va='bottom', fontsize=9)
    delta_cont = res_mut_cont['energy'] - res_wt_cont['energy']
    ax.text(x[1]+width, (res_mut_cont['energy']+res_wt_cont['energy'])/2, f'Δ_cont={delta_cont:+.2f}', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.6))
    plt.tight_layout()
    plt.savefig('vqe_comparison_enhanced_f98y.png', dpi=150)
    plt.show()
    print('OK - vqe_comparison_enhanced_f98y.png')
else:
    print('⚠️ Faltan resultados para gráfico avanzado. Ejecuta primero las celdas anteriores y el notebook base.')
